In [6]:
import numpy as np
import rasterio
from rasterio.transform import from_bounds
from pathlib import Path

from tqdm import tqdm

In [ ]:
# FILL THESE OUT TO MATCH WHERE YOUR DATASETS ARE

raw_data_dir = Path("../embed2heights/raw/")
processed_data_dir = Path("../embed2heights/processed/")
processed_data_dir.mkdir(parents=True, exist_ok=True)

# Alpha earth
We do three things here: 

1. Remove NaNs. For some reason this data has both nodata at -128 AND NaN values, so we set both to zero.
1. Dequantize according to [this methodology](https://developers.google.com/earth-engine/guides/aef_on_gcs_readme).
1. Save as a numpy array. We don't actually need any of the geographic information, and it just makes loading a lot faster.


In [7]:
def dequantize(raw: np.ndarray) -> np.ndarray:
    nodata_mask = np.isnan(raw) | (raw == -128)
    raw = np.nan_to_num(raw, nan=0.0)
    dequantized = ((raw / 127.5) ** 2) * np.sign(raw)
    dequantized[nodata_mask] = 0
    return dequantized

In [8]:
# Training dir
input_dir = raw_data_dir / "train" / "alphaearth_emb"
output_dir = processed_data_dir / "train" / "alphaearth_emb"
output_dir.mkdir(parents=True, exist_ok=True)

for path in tqdm(input_dir.glob("*.tif")):
    with rasterio.open(path) as src:
        raw = src.read().astype(np.float32)
    
    dequantized = dequantize(raw)
    np.save(output_dir / path.stem, dequantized)

2024it [03:31,  9.58it/s]


In [9]:
# Test dir
input_dir = raw_data_dir / "test" / "alphaearth_test_emb"
output_dir = processed_data_dir / "test" / "alphaearth_test_emb"
output_dir.mkdir(parents=True, exist_ok=True)

for path in tqdm(input_dir.glob("*.tif")):
    with rasterio.open(path) as src:
        raw = src.read().astype(np.float32)
    
    dequantized = dequantize(raw)
    np.save(output_dir / path.stem, dequantized)

946it [01:46,  8.89it/s]


# Labels
These are not quantized, but we will save a lot of time converting them to numpy arrays. This data does not have any NaNs or nodata.

In [10]:
input_dir = raw_data_dir / "train" / "labels"
output_dir = processed_data_dir / "train" / "labels"
output_dir.mkdir(parents=True, exist_ok=True)

for path in tqdm(input_dir.glob("*.tif")):
    with rasterio.open(path) as src:
        raw = src.read()
    
    np.save(output_dir / path.stem, raw)

2024it [00:17, 114.06it/s]
